# Loading and Inspecting Medical Images

This notebook demonstrates how to load DICOM and PNG images using the `medical_image` framework.

In [ ]:
!pip install medical-image-std

In [ ]:
# Download sample files
!wget -q -O sample.dcm https://raw.githubusercontent.com/HamzaGbada/dicomPreProcess/master/data/20587054.dcm
!wget -q -O sample.png https://upload.wikimedia.org/wikipedia/commons/thumb/e/e7/Steuben_-_Bataille_de_Poitiers.png/640px-Steuben_-_Bataille_de_Poitiers.png

In [ ]:
import numpy as np
import torch
from medical_image import DicomImage, PNGImage, InMemoryImage, ImageExporter

## 1. Loading a DICOM Image

In [ ]:
dicom = DicomImage("sample.dcm")
dicom.load()

# DICOM images are often uint16 — convert to float for processing
dicom.pixel_data = dicom.pixel_data.float()

print(f"Size: {dicom.width} x {dicom.height}")
print(f"Pixel data type: {dicom.pixel_data.dtype}")
print(f"Device: {dicom.device}")
print(f"Value range: [{dicom.pixel_data.min():.0f}, {dicom.pixel_data.max():.0f}]")

## 2. Loading a PNG Image

In [ ]:
png = PNGImage("sample.png")
png.load()

print(f"Shape: {png.pixel_data.shape}")
print(f"Value range: [{png.pixel_data.min():.0f}, {png.pixel_data.max():.0f}]")

## 3. Creating an Image from a NumPy Array

In [ ]:
# Create a synthetic grayscale image
array = np.random.rand(256, 256).astype(np.float32)

img = DicomImage.from_array(array)
print(f"Size: {img.width} x {img.height}")
print(f"Pixel range: [{img.pixel_data.min():.2f}, {img.pixel_data.max():.2f}]")

## 4. Cloning and Exporting

In [ ]:
# Clone an image (independent deep copy of pixel data)
clone = img.clone()
print(f"Clone size: {clone.width} x {clone.height}")

# Export to PNG
output_path = ImageExporter.save_as(img, format="PNG")
print(f"Saved to: {output_path}")

## 5. Moving Images to GPU

In [ ]:
if torch.cuda.is_available():
    img.to("cuda")
    print(f"Device: {img.device}")
    img.to("cpu")
else:
    print("CUDA not available, staying on CPU")

## 6. Annotations

In [ ]:
from medical_image import Annotation, GeometryType

# Add a bounding box annotation
ann = Annotation(
    shape=GeometryType.RECTANGLE,
    coordinates=[50, 50, 150, 150],
    label="lesion",
)
img.add_annotation(ann)
print(f"Annotations: {len(img.annotations)}")
print(f"Label: {img.annotations[0].label}, Center: {img.annotations[0].center}")

## 7. Patches and Region of Interest

In [ ]:
from medical_image import PatchGrid, RegionOfInterest

# Split image into patches
grid = PatchGrid(img, patch_size=(64, 64))
print(f"Total patches: {len(grid.patches)}")
print(f"Grid shape: {len(grid.grid)} rows x {len(grid.grid[0])} cols")

# Access a single patch
patch = grid.patches[0]
print(f"Patch shape: {patch.pixel_data.shape}")

# Reconstruct the full image from patches
reconstructed = grid.to_image()
print(f"Reconstructed size: {reconstructed.width} x {reconstructed.height}")

In [ ]:
# Extract a region of interest by center coordinates
roi = RegionOfInterest.from_center(img, cx=128, cy=128, half_size=32)
roi_image = roi.load()
print(f"ROI size: {roi_image.width} x {roi_image.height}")